# Day 04：构建 CNN —— 手写 LeNet-5

> 👁️ 第三周 · 视觉的征服 · 第 4 天

前三天我们分别学习了卷积、卷积层和池化。今天，我们要把这些「零件」组装成一个完整的**卷积神经网络（CNN）**。

我们将从零手写 **LeNet-5**——1998 年 Yann LeCun 设计的第一个成功商用的 CNN。它被美国邮政用来识别手写邮政编码，处理了数百万封信件。

**今天的任务**：理解 LeNet-5 的架构设计，用 PyTorch 从零搭建，在 MNIST 上训练并达到 98%+ 的准确率。

---
## 1. 历史背景：CNN 的诞生

### 1998 年：LeNet-5 的诞生

Yann LeCun 在贝尔实验室工作期间，一直在思考一个问题：**如何让计算机自动读取手写数字？**

当时（1990 年代），银行每天要处理数百万张支票，每张支票上的金额都是手写的。人工读取这些数字既慢又容易出错。如果能用计算机自动识别，将节省巨大的人力成本。

LeCun 把卷积、池化和全连接层组合在一起，设计了 LeNet-5。它的架构非常优雅：

```
输入(32×32) → Conv(6@28×28) → Pool(6@14×14) → Conv(16@10×10)
→ Pool(16@5×5) → FC(120) → FC(84) → Output(10)
```

这个架构成为了 CNN 的「标准模板」——卷积提取特征、池化压缩信息、全连接做分类。此后 20 多年的 CNN 架构（AlexNet、VGG、GoogLeNet、ResNet）都是在这个模板上演化而来的。

### LeNet-5 的商业成功

LeNet-5 被部署到美国邮政系统中，用于自动读取手写邮政编码。到 1990 年代末，它每天处理 **10-20%** 的美国邮件——数百万封信件。

这是深度学习第一次大规模商业部署，证明了神经网络不只是学术玩具，而是能解决实际问题的工具。

### LeNet-5 的架构设计哲学

LeNet-5 的设计体现了几个重要的设计原则，这些原则至今仍然适用：

**原则一：逐层抽象**

```
Layer 1 (Conv): 检测边缘、角点（低级特征）
Layer 2 (Pool):  压缩，提供平移不变性
Layer 3 (Conv): 检测形状、部件（中级特征）
Layer 4 (Pool):  进一步压缩
Layer 5-7 (FC):  综合所有特征，做出分类决策
```

每一层都在前一层的基础上提取更抽象的特征。就像认识一个人：先看到轮廓（边缘），再看到五官（形状），最后认出是谁（分类）。

**原则二：通道数递增，空间尺寸递减**

```
空间尺寸: 32 → 28 → 14 → 10 → 5 → 1×1
通道数:   1  → 6  → 6  → 16 → 16 → 120
```

随着网络加深，空间信息逐渐被压缩（通过池化），而特征种类逐渐增多（更多卷积核）。这就像从「看到一棵树」到「看到树叶的纹理」——视野变小了，但细节更丰富了。

**原则三：卷积和池化交替**

```
Conv → Pool → Conv → Pool → FC → FC → Output
```

卷积负责「提取」，池化负责「压缩」。两者交替出现，像呼吸一样——一吸（提取特征）一呼（压缩信息）。

---
## 2. 生活隐喻：流水线工厂

### 隐喻一：汽车制造流水线

LeNet-5 的架构就像一条汽车制造流水线：

| 工位 | LeNet-5 层 | 做什么 |
|------|-----------|--------|
| 冲压车间 | Conv1 (6核) | 从钢板中冲出基本形状（边缘检测） |
| 质检压缩 | Pool1 | 检查关键部位，忽略细节偏差 |
| 焊接车间 | Conv2 (16核) | 把基本形状焊成部件（形状检测） |
| 质检压缩 | Pool2 | 再次检查关键部位 |
| 总装线 | FC1 (120) | 把所有部件组装起来 |
| 终检 | FC2 (84) | 最终质量检查 |
| 出厂标签 | Output (10) | 贴上型号标签（0-9 哪个数字） |

每个工位只做一件事，但环环相扣。前一个工位的输出是后一个工位的输入。任何一个工位出错，最终产品就不合格。

### 隐喻二：侦探破案

CNN 的工作方式也像侦探破案：

1. **Conv1**：侦探到达现场，收集基础线索（指纹、脚印、血迹……）
2. **Pool1**：筛选关键线索，忽略无关细节（邻居的猫脚印不重要）
3. **Conv2**：将线索组合成假设（指纹 + 脚印 → 嫌疑人身高体重）
4. **Pool2**：确认关键假设，排除干扰项
5. **FC**：综合所有证据，得出结论（凶手是谁）

侦探不需要记住现场的每一个灰尘颗粒（池化丢弃了细节），只需要抓住关键证据链。

---
## 3. 数学直觉：LeNet-5 的逐层数据流

### 完整的数据流

让我们追踪一张 MNIST 图片（28×28 灰度）经过 LeNet-5 的完整旅程：

```
输入: [1, 1, 28, 28]  ← [batch, channel, height, width]
         │
    ┌────▼────┐
    │ Conv1   │  nn.Conv2d(1, 6, 5)  → 6个5×5卷积核
    │ [1,6,24,24]│  输出: 6张 24×24 的特征图
    └────┬────┘
         │
    ┌────▼────┐
    │ Pool1   │  nn.MaxPool2d(2, 2)  → 2×2窗口, stride=2
    │ [1,6,12,12]│  输出: 6张 12×12 的特征图（尺寸减半）
    └────┬────┘
         │
    ┌────▼────┐
    │ Conv2   │  nn.Conv2d(6, 16, 5) → 16个5×5卷积核
    │ [1,16,8,8]│  输出: 16张 8×8 的特征图
    └────┬────┘
         │
    ┌────▼────┐
    │ Pool2   │  nn.MaxPool2d(2, 2)
    │ [1,16,4,4]│  输出: 16张 4×4 的特征图
    └────┬────┘
         │
    ┌────▼────┐
    │ Flatten │  展平成一维向量
    │ [1, 256] │  16 × 4 × 4 = 256
    └────┬────┘
         │
    ┌────▼────┐
    │ FC1     │  nn.Linear(256, 120)
    │ [1, 120] │
    └────┬────┘
         │
    ┌────▼────┐
    │ FC2     │  nn.Linear(120, 84)
    │ [1, 84]  │
    └────┬────┘
         │
    ┌────▼────┐
    │ Output  │  nn.Linear(84, 10)
    │ [1, 10]  │  10 个类别的得分
    └─────────┘
```

### 参数量统计

| 层 | 参数量 | 计算方式 |
|----|--------|---------|
| Conv1 | 156 | 6×(1×5×5) + 6(bias) |
| Conv2 | 2,416 | 16×(6×5×5) + 16(bias) |
| FC1 | 30,720 | 256×120 |
| FC2 | 10,164 | 120×84 + 84(bias) |
| Output | 850 | 84×10 + 10(bias) |
| **总计** | **44,306** | |

注意：**全连接层占了绝大多数参数**（FC1 的 30,720 个参数占了总数的 69%）。这就是为什么现代 CNN 倾向于减少甚至消除全连接层（用全局平均池化替代）。

---
## 4. 代码实现：从零搭建 LeNet-5

下面用 PyTorch 从零搭建 LeNet-5。每个代码 cell 都是独立的。

In [ ]:
# 设置中文字体
import matplotlib.pyplot as plt
try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

import torch
import torch.nn as nn
import torch.nn.functional as F

class LeNet5(nn.Module):
    """LeNet-5 的 PyTorch 实现。

    架构: Conv1 → Pool1 → Conv2 → Pool2 → FC1 → FC2 → Output
    输入: [N, 1, 28, 28]  输出: [N, 10]
    """
    def __init__(self):
        super(LeNet5, self).__init__()

        # 卷积层 1: 1通道 → 6通道, 5×5卷积核
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)
        # 池化层 1: 2×2 最大池化
        self.pool1 = nn.MaxPool2d(2, 2)

        # 卷积层 2: 6通道 → 16通道, 5×5卷积核
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        # 池化层 2: 2×2 最大池化
        self.pool2 = nn.MaxPool2d(2, 2)

        # 全连接层
        self.fc1 = nn.Linear(16 * 4 * 4, 120)  # 16通道 × 4×4 = 256
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)  # 10 个数字类别

    def forward(self, x):
        # 第一个卷积-池化块
        x = self.pool1(F.relu(self.conv1(x)))  # [N, 6, 12, 12]

        # 第二个卷积-池化块
        x = self.pool2(F.relu(self.conv2(x)))  # [N, 16, 4, 4]

        # 展平
        x = x.view(-1, 16 * 4 * 4)  # [N, 256]

        # 全连接层
        x = F.relu(self.fc1(x))  # [N, 120]
        x = F.relu(self.fc2(x))  # [N, 84]
        x = self.fc3(x)          # [N, 10]  原始得分（logits）

        return x

# 创建模型并查看结构
model = LeNet5()
print("LeNet-5 模型结构：")
print(model)
print(f"\n总参数量: {sum(p.numel() for p in model.parameters()):,}")

# 测试前向传播
dummy_input = torch.randn(1, 1, 28, 28)  # 模拟一张 MNIST 图片
output = model(dummy_input)
print(f"\n输入形状: {dummy_input.shape}")
print(f"输出形状: {output.shape}  (10个类别的得分)")

### 加载 MNIST 数据集

MNIST 是手写数字识别的「Hello World」——7 万张 28×28 的灰度图片，10 个类别（0-9）。

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

# 数据预处理：转成 Tensor，归一化到 [0, 1]
transform = transforms.Compose([
    transforms.ToTensor(),
])

# 下载 MNIST 数据集
train_dataset = datasets.MNIST(
    root='Week03/data', train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    root='Week03/data', train=False, download=True, transform=transform
)

# 创建 DataLoader
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"训练集: {len(train_dataset)} 张图片")
print(f"测试集: {len(test_dataset)} 张图片")
print(f"类别: 0-9 (共 10 类)")
print(f"图片尺寸: 28×28 灰度")

# 可视化几张样本
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    img, label = train_dataset[i]
    ax.imshow(img[0].numpy(), cmap='gray')
    ax.set_title(f'数字: {label}')
    ax.axis('off')
plt.suptitle('MNIST 样本展示', fontsize=14)
plt.tight_layout()
plt.savefig('Week03/images/cnn_day04_mnist_samples.png', dpi=150, bbox_inches='tight')
plt.show()

### 训练 LeNet-5

用标准的训练流程：前向传播 → 计算损失 → 反向传播 → 更新参数。

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

# 定义 LeNet-5
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 4 * 4)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# 加载数据
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST('Week03/data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('Week03/data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 创建模型、损失函数、优化器
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LeNet5().to(device)
criterion = nn.CrossEntropyLoss()  # 交叉熵损失（分类任务标配）
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # Adam 优化器

print(f"使用设备: {device}")
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")
print("\n开始训练...")

# 训练循环
epochs = 5
train_losses = []
train_accs = []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # 前向传播
        outputs = model(images)
        loss = criterion(outputs, labels)

        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # 统计
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)

    print(f"Epoch [{epoch+1}/{epochs}]  Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.2f}%")

print("\n训练完成！")

# 绘制训练曲线
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, epochs+1), train_losses, 'b-o')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('训练损失曲线')
ax1.grid(True)

ax2.plot(range(1, epochs+1), train_accs, 'r-o')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('训练准确率曲线')
ax2.grid(True)

plt.suptitle('LeNet-5 训练过程', fontsize=14)
plt.tight_layout()
plt.savefig('Week03/images/cnn_day04_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 测试模型

在测试集上评估模型的泛化能力。

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

# 定义 LeNet-5
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 4 * 4)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# 加载数据
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST('Week03/data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('Week03/data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 训练模型
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LeNet5().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# 测试
model.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = 100 * correct / total
print(f"测试集准确率: {test_acc:.2f}%")
print(f"正确: {correct} / {total}")

# 可视化一些预测结果
fig, axes = plt.subplots(2, 5, figsize=(12, 6))
test_images = []
test_true = []

# 取前 10 张测试图片
for i in range(10):
    img, label = test_dataset[i]
    test_images.append(img)
    test_true.append(label)

for i, ax in enumerate(axes.flatten()):
    img = test_images[i]
    true_label = test_true[i]

    # 预测
    img_input = img.unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(img_input)
        pred_label = torch.argmax(output, dim=1).item()

    ax.imshow(img[0].numpy(), cmap='gray')
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(f'真实: {true_label}  预测: {pred_label}', color=color)
    ax.axis('off')

plt.suptitle(f'测试结果 (准确率: {test_acc:.1f}%)', fontsize=14)
plt.tight_layout()
plt.savefig('Week03/images/cnn_day04_test_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. 验证实验：CNN vs MLP

用相同的 MNIST 任务对比 CNN（LeNet-5）和 MLP 的性能差异。

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 加载数据
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST('Week03/data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('Week03/data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# MLP 模型（和 LeNet-5 参数量相近）
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(28*28, 300)
        self.fc2 = nn.Linear(300, 100)
        self.fc3 = nn.Linear(100, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)  # 拍扁
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# LeNet-5
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(16*4*4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.view(-1, 16*4*4)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

def train_and_eval(model, name):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(5):
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    params = sum(p.numel() for p in model.parameters())
    print(f"{name}: 准确率 = {acc:.2f}%, 参数量 = {params:,}")
    return acc, params

print("CNN vs MLP 对比实验")
print("=" * 50)
cnn_acc, cnn_params = train_and_eval(LeNet5(), "LeNet-5 (CNN)")
mlp_acc, mlp_params = train_and_eval(MLP(), "MLP")
print(f"\n结论：CNN 用更少的参数，达到了更高的准确率")
print(f"  CNN: {cnn_params:,} 参数 → {cnn_acc:.1f}%")
print(f"  MLP: {mlp_params:,} 参数 → {mlp_acc:.1f}%")

---
## 翻译词典

| 生活直觉 | 深度学习术语 |
|----------|-------------|
| 汽车流水线 | CNN 的层级架构 |
| 冲压车间 | 第一个卷积层（提取基础特征） |
| 质检压缩 | 池化层（压缩 + 平移不变性） |
| 总装线 | 全连接层（综合特征做决策） |
| 出厂标签 | 输出层（分类结果） |
| 侦探收集线索 | 卷积提取特征 |
| 筛选关键线索 | 池化压缩信息 |

---
## 今日总结

| 概念 | 直觉理解 |
|------|----------|
| LeNet-5 | 第一个成功商用的 CNN，Conv→Pool→Conv→Pool→FC |
| 架构设计 | 空间尺寸递减，通道数递增 |
| Conv + Pool | 卷积提取特征，池化压缩信息 |
| MNIST | 手写数字识别的标准基准数据集 |
| CNN vs MLP | CNN 用更少参数达到更高准确率 |

**LeNet-5 的设计原则**：
- 逐层抽象：边缘 → 形状 → 部件 → 物体
- 通道递增：6 → 16 → 120（特征越来越丰富）
- 空间递减：28 → 14 → 10 → 5（位置越来越模糊）
- 卷积+池化交替：提取 → 压缩 → 提取 → 压缩

**明日预告**：LeNet-5 只有 5 层，训练很顺利。但如果把网络加深到 20 层、50 层呢？你会发现一个诡异的现象——**网络越深，效果反而越差**。明天学习**梯度消失**——深度网络的「老年痴呆」，以及如何用 ReLU 来缓解它。